WhatNet  –  Thai


In [ ]:
#  Thai script is an abugida with distinctive circular loops at the top of many
#  consonants (e.g., ก, ข, ค).  Burapha-TH covers 44 consonants plus digit
#  classes.  Key structural differences vs. Kuzushiji:
#
#    num_classes   : 68   (68 character)
#    image_size    : 64   (Burapha-TH is natively 64×64)
#    Scaffold bypass: REMOVED – Thai glyphs have complex embedded loops that
#                        a simple bypass convolution would smear rather than
#                        disentangle.  The full encoder is used end-to-end.
#    Decoder Head
#    Multi-scale GAP
#    Gated Dense Head
#    Classification Head
#    Augmentation  : rotation ±8° KEPT (Thai handwriting varies in angle)
#                       random_flip REMOVED – Thai script is directional
#    AFC           : OPTIONAL – 54 classes is manageable without it;
#                       included but with a lightweight capsule_dim=8

#  Dataset
#  Burapha-TH  (Thai handwritten character dataset, Burapha University)
#    • ~60,000 64×64 greyscale images
#    • 68 characters
#    • Expected layout in CFG['data_dir']:
#        thai_train.npz  –  arr_0: uint8 (N, 64, 64),  arr_1: uint8 (N,)
#        thai_test.npz   –  arr_0: uint8 (M, 64, 64),  arr_1: uint8 (M,)
#      OR split files:
#        thai-train-imgs.npz / thai-train-lbls.npz
#        thai-test-imgs.npz  / thai-test-lbls.npz
#    • Kaggle path (if using the Burapha-TH Kaggle dataset):
#        "/kaggle/input/burapha-th/"

In [ ]:
# importing necessary dependencies
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#  1. CONFIGURATION
CFG = {
    # Will be auto-detected from the number of class folders found on disk.
    # Override here only if you want to force a specific value.
    "num_classes":      68,       # None = auto-detect
    "image_size":       64,
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # Root folder that contains "train/" and "test/" sub-directories.
    # Kaggle path example: "/kaggle/input/thai-dataset"
    "data_dir":    "/kaggle/input/datasets/rautranjit/thai-dataset",

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG     = CFG["image_size"]
BS      = CFG["batch_size"]
AUTOTUNE = tf.data.AUTOTUNE

#  2. DATASET PIPELINE  (folder-image loader)
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

# Auto-detect num_classes from folder count
_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes  = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

if CFG["num_classes"] is None:
    NUM_CLASSES = len(_train_classes)
    print(f"[INFO] Auto-detected {NUM_CLASSES} classes from train folder.")
else:
    NUM_CLASSES = CFG["num_classes"]
    print(f"[INFO] Using CFG num_classes = {NUM_CLASSES}.")

if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")

if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}).  Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

# Helper: load a split from an image folder
def load_folder_split(root: str, class_names: list) -> tuple:
    """
    Walk root/<class_name>/* and return (images_uint8, labels_int32).

    Images are resized to (IMG, IMG) and converted to greyscale.
    Labels are the sorted folder index (0-based).
    """
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    images, labels = [], []

    for cls in class_names:
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            print(f"[WARN] Missing class folder: {cls_dir}")
            continue
        idx = class_to_idx[cls]
        for fname in sorted(os.listdir(cls_dir)):
            fpath = os.path.join(cls_dir, fname)
            if not os.path.isfile(fpath):
                continue
            ext = os.path.splitext(fname)[1].lower()
            if ext not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
                continue
            try:
                raw    = tf.io.read_file(fpath)
                # decode_image handles PNG / JPEG / BMP / GIF automatically
                img    = tf.image.decode_image(raw, channels=1, expand_animations=False)
                img    = tf.image.resize(img, [IMG, IMG])
                img    = tf.cast(img, tf.uint8)
                images.append(img.numpy())
                labels.append(idx)
            except Exception as e:
                print(f"[WARN] Could not read {fpath}: {e}")

    images_np = np.array(images, dtype=np.uint8)   # (N, IMG, IMG, 1)
    labels_np = np.array(labels, dtype=np.int32)
    return images_np, labels_np

print("[INFO] Loading train images …  (this may take a moment)")
x_train_all, y_train_all = load_folder_split(train_dir, _train_classes)
print(f"[INFO] Train loaded: {len(x_train_all):,} images")

print("[INFO] Loading test images …")
x_test, y_test = load_folder_split(test_dir, _train_classes)
print(f"[INFO] Test  loaded: {len(x_test):,} images")

# Squeeze channel dim for val split (re-added later)
# images are currently (N, IMG, IMG, 1)  – squeeze to (N, IMG, IMG) for split
x_train_all = x_train_all[..., 0]   # → (N, IMG, IMG)
x_test      = x_test[..., 0]         # → (N, IMG, IMG)

# Train / val split
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# Add channel dim: (N,IMG,IMG) → (N,IMG,IMG,1)
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# Rotation augmentation
_HAS_RANDOM_ROTATION = hasattr(tf.keras.layers, "RandomRotation")
if _HAS_RANDOM_ROTATION:
    print("[INFO] RandomRotation available – rotation augmentation enabled (±8°).")
    _rotator = tf.keras.layers.RandomRotation(
        factor=8/360,
        fill_mode="constant",
        fill_value=-1.0,
        seed=SEED,
    )
else:
    print("[WARN] RandomRotation not available – rotation augmentation disabled.")
    _rotator = None

# Preprocessing helpers
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Stochastic augmentation – training only.

    Thai consonants have:
      • Circular loops that sit at varying heights/positions
      • Diacritics (sara) that hover above or below the baseline
      • Significant pen-angle variation in handwriting

    Augmentations:
      • Brightness / contrast jitter – ink variation
      • Pad-then-random-crop         – translation ±4 px (64-px image)
      • Random rotation ±8°          – writing angle variation

    No flip: Thai script is strongly directional.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    # ±4 px translation on 64-px images (pad=4 each side → crop back to 64)
    img = tf.pad(img, [[4, 4], [4, 4], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    if _rotator is not None:
        img = _rotator(tf.expand_dims(img, 0))[0]
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# Build tf.data pipelines
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

#  3. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  4. BUILDING BLOCKS
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 8):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

#  5. MODEL DEFINITION
def build_whatnet_thai(num_classes: int, image_size: int = 64, head_units: int = 256,) -> Model:
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # Stem: triple-path (3×3, 1×5, 5×1)
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t)
    t = layers.Activation(gelu)(t)

    h = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Activation(gelu)(h)

    v = layers.Conv2D(32, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v)
    v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])          # 96 channels
    stem = channel_attention(stem, 96)
    stem = layers.Conv2D(96, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # Encoder (no scaffold bypass)
    enc1 = dense_res_block(stem, 96,  96)   # → 32×32
    enc2 = dense_res_block(enc1, 96,  192)  # → 16×16
    enc3 = dense_res_block(enc2, 192, 384)  # →  8× 8

    # Decoder
    # Multi-scale GAP fusion
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    # Adaptive Filter Capsule (AFC)
    # Projects the fused multi-scale vector into capsule space.
    # Each of the K capsules learns to respond to one character class.
    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # Classification head
    # Dense projection of the raw GAP features (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # Gated fusion: AFC scores + dense-head logits
    # A learned scalar gate (per-sample softmax over 2 weights) blends the
    # AFC capsule scores with the plain dense logits.  This lets the model
    # learn how much to trust the capsule routing vs. the direct projection.
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)

    # gate[:,0] weights the dense head; gate[:,1] weights the AFC output
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out,gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])

    model = keras.Model(inputs=inp, outputs=outputs, name="WhatNet-Thai")
    return model

MODELS_TF = {
    "WhatNet-Thai": lambda: build_whatnet_thai(NUM_CLASSES, IMG),
}

#  6. LR SCHEDULE
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

#  7. TRAINING & EVALUATION HELPERS
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

#  8. TRAIN + EVALUATE
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

#  9. FINAL TEST-SET EVALUATION
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

#  10. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

# Method 2

In [ ]:
#  WhatNet  –  Thai Script Recognition  (PyTorch)
import os, time, random, json, math
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import torchvision.transforms as T

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

#  1. CONFIGURATION
CFG = {
    "num_classes":      68,
    "image_size":       64,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # Root folder that contains "train/" and "test/" sub-directories.
    "data_dir":    "/kaggle/input/datasets/rautranjit/thai-dataset",

    "results_dir": "./results",
    "num_workers": 4,
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG = CFG["image_size"]
BS  = CFG["batch_size"]

#  2. DATASET
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

NUM_CLASSES = CFG["num_classes"] or len(_train_classes)
if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")
if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}). Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] NUM_CLASSES = {NUM_CLASSES}")
print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

class ThaiScriptDataset(Dataset):
    """
    Folder-based dataset for Thai script images.
    Loads grayscale images resized to (IMG x IMG).
    """
    def __init__(self, root: str, class_names: list, transform=None):
        self.transform   = transform
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples     = []  # list of (path, label_int)

        for cls in class_names:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                print(f"[WARN] Missing class folder: {cls_dir}")
                continue
            idx = self.class_to_idx[cls]
            for fname in sorted(os.listdir(cls_dir)):
                fpath = os.path.join(cls_dir, fname)
                if not os.path.isfile(fpath):
                    continue
                if os.path.splitext(fname)[1].lower() not in VALID_EXTS:
                    continue
                self.samples.append((fpath, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        fpath, label = self.samples[index]
        try:
            img = Image.open(fpath).convert("L")          # grayscale
            img = img.resize((IMG, IMG), Image.BILINEAR)
        except Exception as e:
            print(f"[WARN] Could not read {fpath}: {e}")
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms
# Normalise to [-1, 1]  (same as TF version: x/127.5 - 1)
_mean = (0.5,)
_std  = (0.5,)

train_transform = T.Compose([
    T.RandomAffine(
        degrees=8,                          # ±8° rotation
        translate=(4/IMG, 4/IMG),           # ±4 px translation
    ),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),                           # [0, 1]
    T.Normalize(_mean, _std),              #[-1, 1]
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(_mean, _std),
])

# Build datasets & loaders
print("[INFO] Building dataset objects …")
full_train_ds = ThaiScriptDataset(train_dir, _train_classes, transform=None)
test_ds_raw   = ThaiScriptDataset(test_dir,  _train_classes, transform=None)

n_total = len(full_train_ds)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

g = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(full_train_ds, [n_train, n_val], generator=g)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds_raw):,}")

# Wrap subsets to apply different transforms
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        fpath, label = self.subset.dataset.samples[self.subset.indices[idx]]
        try:
            img = Image.open(fpath).convert("L").resize((IMG, IMG), Image.BILINEAR)
        except Exception:
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        return self.transform(img), label

train_dataset = TransformSubset(train_subset, train_transform)
val_dataset   = TransformSubset(val_subset,   eval_transform)

class SimpleDataset(Dataset):
    def __init__(self, base_ds, transform):
        self.base = base_ds
        self.transform = transform
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        fpath, label = self.base.samples[idx]
        try:
            img = Image.open(fpath).convert("L").resize((IMG, IMG), Image.BILINEAR)
        except Exception:
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        return self.transform(img), label

test_dataset = SimpleDataset(test_ds_raw, eval_transform)

def make_loader(ds, shuffle):
    return DataLoader(
        ds,
        batch_size=BS,
        shuffle=shuffle,
        num_workers=CFG["num_workers"],
        pin_memory=True,
        drop_last=False,
    )

train_loader = make_loader(train_dataset, shuffle=True)
val_loader   = make_loader(val_dataset,   shuffle=False)
test_loader  = make_loader(test_dataset,  shuffle=False)

#  3. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: nn.Module, name: str) -> None:
    W         = 62
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_train = total - trainable
    title     = f"  {name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_train:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  4. BUILDING BLOCKS
class ResidualBlock(nn.Module):
    """Two Conv-BN-GELU layers with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        res = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.gelu(x + res)
        return x


class DenseResBlock(nn.Module):
    """
    Three stacked ResidualBlocks whose outputs are concatenated,
    fused with a 1×1 conv, then downsampled 2× with a depthwise stride-2 conv.
    Optionally adapts the channel count via a projection shortcut.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # Channel projection if needed
        self.proj = None
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        self.r1 = ResidualBlock(out_channels)
        self.r2 = ResidualBlock(out_channels)
        self.r3 = ResidualBlock(out_channels)

        # Fuse concatenated (3×out_channels) back to out_channels
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * 3, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        # Depthwise stride-2 downsampling
        self.down = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, stride=2,
                      padding=1, groups=out_channels, bias=False),
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))

        r1 = self.r1(x)
        r2 = self.r2(r1)
        r3 = self.r3(r2)

        cat = torch.cat([r1, r2, r3], dim=1)           # (B, 3C, H, W)
        out = F.gelu(self.fuse(cat))                    # (B, C, H, W)
        out = F.gelu(self.down(out))                    # (B, C, H/2, W/2)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        attn = self.fc(self.gap(x)).unsqueeze(-1).unsqueeze(-1)
        return x * attn


class AdaptiveFilterCapsule(nn.Module):
    """
    Projects the fused multi-scale vector into a K-dimensional capsule space.
    Each of the K outputs learns to respond to one character class.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 8):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_features, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B       = x.size(0)
        h       = F.gelu(self.fc1(x))
        h       = self.fc2(h).view(B, self.num_classes, self.capsule_dim)  # (B, K, d)
        # Broadcast x into capsule space and element-wise multiply
        x_exp   = x.unsqueeze(1).expand(B, self.num_classes, -1)           # (B, K, F)
        x_sl    = x_exp[:, :, :self.capsule_dim]                           # (B, K, d)
        caps    = (x_sl * h).sum(dim=-1)                                   # (B, K)
        caps    = self.bn(caps)
        return caps

#  5. MODEL DEFINITION
class WhatNetThai(nn.Module):
    """
    WhatNet-Thai  –  PyTorch port.

    Architecture:
      Stem : three parallel convolutions (3×3, 1×5, 5×1) → channel attention
      Enc1 : DenseResBlock 96  → 32×32
      Enc2 : DenseResBlock 192 → 16×16
      Enc3 : DenseResBlock 384 →  8× 8
      Head : multi-scale GAP fusion → AFC + dense head → gated logit blend
    """
    def __init__(self, num_classes: int, image_size: int = 64,
                 head_units: int = 256):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_t = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 32, (1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 32, (5, 1), padding=(2, 0), bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_attn = ChannelAttention(96)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(96, 96, 1, bias=False),
            nn.BatchNorm2d(96), nn.GELU(),
        )

        # Encoder
        self.enc1 = DenseResBlock(96,  96)    # → 32×32
        self.enc2 = DenseResBlock(96,  192)   # → 16×16
        self.enc3 = DenseResBlock(192, 384)   # →  8× 8

        # Fused feature dim = 96 + 192 + 384 = 672
        F_DIM = 96 + 192 + 384

        # Adaptive Filter Capsule
        self.afc = AdaptiveFilterCapsule(F_DIM, K)

        # Dense head
        self.head_dense  = nn.Linear(F_DIM, head_units, bias=False)
        self.head_ln     = nn.LayerNorm(head_units)
        self.head_logits = nn.Linear(head_units, K)

        # Gated fusion
        self.gate = nn.Linear(K * 2, 2)

    def forward(self, x):
        # Stem
        t    = self.stem_t(x)
        h    = self.stem_h(x)
        v    = self.stem_v(x)
        stem = torch.cat([t, h, v], dim=1)          # (B, 96, H, W)
        stem = self.stem_attn(stem)
        stem = self.stem_proj(stem)

        # Encoder
        e1 = self.enc1(stem)                         # (B,  96, 32, 32)
        e2 = self.enc2(e1)                           # (B, 192, 16, 16)
        e3 = self.enc3(e2)                           # (B, 384,  8,  8)

        # Multi-scale GAP
        gap1 = e1.mean(dim=(-2, -1))                 # (B,  96)
        gap2 = e2.mean(dim=(-2, -1))                 # (B, 192)
        gap3 = e3.mean(dim=(-2, -1))                 # (B, 384)
        fused = torch.cat([gap1, gap2, gap3], dim=1) # (B, 672)

        # AFC branch
        afc_out = self.afc(fused)                    # (B, K)

        # Dense head branch
        x_ = F.gelu(self.head_ln(self.head_dense(fused)))
        x_ = self.head_logits(x_)                   # (B, K)

        # Gated fusion
        combined = torch.cat([x_, afc_out], dim=1)  # (B, 2K)
        gate     = F.softmax(self.gate(combined), dim=1)  # (B, 2)

        x_scaled   = x_      * gate[:, 0:1]
        afc_scaled = afc_out * gate[:, 1:2]
        logits     = x_scaled + afc_scaled            # (B, K)
        return logits

#  6. LR SCHEDULE  (cosine annealing)
def cosine_annealing_lambda(total_steps: int, base_lr: float):
    """Returns a lambda for LambdaLR that replicates the TF CosineAnnealing."""
    def lr_lambda(current_step: int):
        cosine = 0.5 * (1.0 + math.cos(math.pi * current_step / total_steps))
        return max(cosine, 1e-6 / base_lr)
    return lr_lambda

#  7. TRAINING & EVALUATION HELPERS
def compute_macro_f1(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            preds  = model(images).argmax(dim=1).cpu().numpy()
            lbls   = labels.numpy()
            for c in range(NUM_CLASSES):
                tp[c] += np.sum((preds == c) & (lbls == c))
                fp[c] += np.sum((preds == c) & (lbls != c))
                fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

def evaluate(model: nn.Module, loader: DataLoader,
             criterion: nn.Module) -> tuple:
    """Returns (avg_loss, accuracy_pct)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits  = model(images)
            loss    = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            preds   = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
    return total_loss / total, correct / total * 100.0

#  8. TRAINING LOOP
steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

_COL_BAR_W = 20

def train_one_epoch(model, loader, optimizer, scheduler, criterion, epoch, total_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total * 100.0


def train_model(model: nn.Module, name: str) -> dict:
    model = model.to(DEVICE)
    print_model_summary(model, name)

    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    optimizer = AdamW(model.parameters(),
                      lr=CFG["lr"],
                      weight_decay=CFG["weight_decay"])
    scheduler = LambdaLR(optimizer, cosine_annealing_lambda(total_steps, CFG["lr"]))

    best_val_acc   = 0.0
    best_state     = None
    patience       = 15
    no_improve     = 0
    history        = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0 = time.time()

    for epoch in range(CFG["epochs"]):
        ep_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            epoch, CFG["epochs"])
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc / 100.0)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc / 100.0)

        # Progress bar
        ep_num  = epoch + 1
        pct     = ep_num / CFG["epochs"]
        filled  = int(_COL_BAR_W * pct)
        bar     = "█" * filled + "░" * (_COL_BAR_W - filled)
        lr_val  = scheduler.get_last_lr()[0]
        elapsed = time.time() - ep_start
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{CFG[\"epochs\"]}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={train_loss:.4f}', 'white')}  "
            f"{_c(f'acc={train_acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < train_acc else 'green')}  "
            f"{_c(f'lr={lr_val:.2e}', 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

        # Checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            ckpt_path    = os.path.join(CFG["results_dir"], f"{name}_best.pt")
            torch.save(best_state, ckpt_path)
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  {_c('Early stopping triggered.', 'yellow')} "
                      f"(no val_acc improvement for {patience} epochs)")
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - t0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val_acc:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    return model, history

#  9. RUN
MODELS = {
    "WhatNet-Thai": lambda: WhatNetThai(NUM_CLASSES, IMG),
}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

trained_models = {}
all_histories  = {}

for name, model_fn in MODELS.items():
    model, history = train_model(model_fn(), name)
    trained_models[name] = model
    all_histories[name]  = history

#  10. FINAL TEST-SET EVALUATION
criterion_eval = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
results = {}

for name, model in trained_models.items():
    test_loss, test_acc = evaluate(model, test_loader, criterion_eval)
    macro_f1            = compute_macro_f1(model, test_loader)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    sum(p.numel() for p in model.parameters()),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

#  11. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

In [2]:
#  WhatNet  –  Thai Script Recognition  (PyTorch)
import os, time, random, json, math, zipfile, shutil, re
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import torchvision.transforms as T

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

#  1. CONFIGURATION
CFG = {
    "num_classes":      68,
    "image_size":       64,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # Path to the zip file
    "data_zip":    "/teamspace/studios/this_studio/thai-dataset.zip",
    "extract_dir": "/teamspace/studios/this_studio/thai-dataset-extracted",

    "results_dir": "./results",
    "num_workers": 4,
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG = CFG["image_size"]
BS  = CFG["batch_size"]

#  2. DATASET – unzip and locate train/test
def extract_dataset(zip_path: str, extract_to: str):
    """
    Unzips the dataset and returns (train_dir, test_dir).
    Handles a top-level wrapper folder inside the zip automatically.
    """
    if os.path.exists(extract_to):
        shutil.rmtree(extract_to)
    os.makedirs(extract_to, exist_ok=True)

    print(f"[INFO] Extracting {zip_path} → {extract_to} …")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    print("[INFO] Extraction complete.")

    # Walk down any single top-level wrapper folder
    root = extract_to
    while True:
        entries = [e for e in os.listdir(root)
                   if not e.startswith("__") and not e.startswith(".")]
        if len(entries) == 1 and os.path.isdir(os.path.join(root, entries[0])):
            root = os.path.join(root, entries[0])
        else:
            break

    train_dir = os.path.join(root, "train")
    test_dir  = os.path.join(root, "test")

    for d in (train_dir, test_dir):
        if not os.path.isdir(d):
            raise FileNotFoundError(
                f"[ERROR] Expected directory not found after extraction: {d}\n"
                f"Zip contents root: {root}\n"
                f"Found: {os.listdir(root)}"
            )
    return train_dir, test_dir


train_dir, test_dir = extract_dataset(CFG["data_zip"], CFG["extract_dir"])
print(f"[INFO] train_dir = {train_dir}")
print(f"[INFO] test_dir  = {test_dir}")


def _list_class_folders(directory: str) -> list:
    """Return sorted class folder names, e.g. '00-161-A1-KO KAI'."""
    return sorted([
        c for c in os.listdir(directory)
        if os.path.isdir(os.path.join(directory, c))
    ])

def _short_label(folder_name: str) -> str:
    """'00-161-A1-KO KAI' → 'KO KAI'"""
    m = re.match(r"^\d+-\d+-\w+-(.+)$", folder_name)
    return m.group(1) if m else folder_name


_train_classes = _list_class_folders(train_dir)
_test_classes  = _list_class_folders(test_dir)
short_labels   = [_short_label(c) for c in _train_classes]

NUM_CLASSES = CFG["num_classes"] or len(_train_classes)
if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")
if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}). Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] NUM_CLASSES = {NUM_CLASSES}")
print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")
print(f"[INFO] Short labels (first 5): {short_labels[:5]} …")

VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


class ThaiScriptDataset(Dataset):
    """
    Folder-based dataset for Thai script images.
    Loads grayscale images resized to (IMG x IMG).
    """
    def __init__(self, root: str, class_names: list, transform=None):
        self.transform    = transform
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples      = []  # list of (path, label_int)

        for cls in class_names:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                print(f"[WARN] Missing class folder: {cls_dir}")
                continue
            idx = self.class_to_idx[cls]
            for fname in sorted(os.listdir(cls_dir)):
                fpath = os.path.join(cls_dir, fname)
                if not os.path.isfile(fpath):
                    continue
                if os.path.splitext(fname)[1].lower() not in VALID_EXTS:
                    continue
                self.samples.append((fpath, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        fpath, label = self.samples[index]
        try:
            img = Image.open(fpath).convert("L")          # grayscale
            img = img.resize((IMG, IMG), Image.BILINEAR)
        except Exception as e:
            print(f"[WARN] Could not read {fpath}: {e}")
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        if self.transform:
            img = self.transform(img)
        return img, label


# Transforms  (normalise to [-1, 1])
_mean = (0.5,)
_std  = (0.5,)

train_transform = T.Compose([
    T.RandomAffine(
        degrees=8,
        translate=(4/IMG, 4/IMG),
    ),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(_mean, _std),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(_mean, _std),
])

# Build datasets & loaders
print("[INFO] Building dataset objects …")
full_train_ds = ThaiScriptDataset(train_dir, _train_classes, transform=None)
test_ds_raw   = ThaiScriptDataset(test_dir,  _train_classes, transform=None)

n_total = len(full_train_ds)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

g = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(full_train_ds, [n_train, n_val], generator=g)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds_raw):,}")


class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        fpath, label = self.subset.dataset.samples[self.subset.indices[idx]]
        try:
            img = Image.open(fpath).convert("L").resize((IMG, IMG), Image.BILINEAR)
        except Exception:
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        return self.transform(img), label


class SimpleDataset(Dataset):
    def __init__(self, base_ds, transform):
        self.base      = base_ds
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        fpath, label = self.base.samples[idx]
        try:
            img = Image.open(fpath).convert("L").resize((IMG, IMG), Image.BILINEAR)
        except Exception:
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        return self.transform(img), label


train_dataset = TransformSubset(train_subset, train_transform)
val_dataset   = TransformSubset(val_subset,   eval_transform)
test_dataset  = SimpleDataset(test_ds_raw,    eval_transform)


def make_loader(ds, shuffle):
    return DataLoader(
        ds,
        batch_size=BS,
        shuffle=shuffle,
        num_workers=CFG["num_workers"],
        pin_memory=True,
        drop_last=False,
    )

train_loader = make_loader(train_dataset, shuffle=True)
val_loader   = make_loader(val_dataset,   shuffle=False)
test_loader  = make_loader(test_dataset,  shuffle=False)

#  3. DISPLAY UTILITIES
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: nn.Module, name: str) -> None:
    W         = 62
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_train = total - trainable
    title     = f"  {name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_train:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  4. BUILDING BLOCKS
class ResidualBlock(nn.Module):
    """Two Conv-BN-GELU layers with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        res = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.gelu(x + res)
        return x


class DenseResBlock(nn.Module):
    """
    Three stacked ResidualBlocks whose outputs are concatenated,
    fused with a 1×1 conv, then downsampled 2× with a depthwise stride-2 conv.
    Optionally adapts the channel count via a projection shortcut.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.proj = None
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        self.r1 = ResidualBlock(out_channels)
        self.r2 = ResidualBlock(out_channels)
        self.r3 = ResidualBlock(out_channels)

        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * 3, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        self.down = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, stride=2,
                      padding=1, groups=out_channels, bias=False),
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))

        r1 = self.r1(x)
        r2 = self.r2(r1)
        r3 = self.r3(r2)

        cat = torch.cat([r1, r2, r3], dim=1)
        out = F.gelu(self.fuse(cat))
        out = F.gelu(self.down(out))
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        attn = self.fc(self.gap(x)).unsqueeze(-1).unsqueeze(-1)
        return x * attn


class AdaptiveFilterCapsule(nn.Module):
    """
    Projects the fused multi-scale vector into a K-dimensional capsule space.
    Each of the K outputs learns to respond to one character class.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 8):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_features, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B     = x.size(0)
        h     = F.gelu(self.fc1(x))
        h     = self.fc2(h).view(B, self.num_classes, self.capsule_dim)
        x_exp = x.unsqueeze(1).expand(B, self.num_classes, -1)
        x_sl  = x_exp[:, :, :self.capsule_dim]
        caps  = (x_sl * h).sum(dim=-1)
        caps  = self.bn(caps)
        return caps

#  5. MODEL DEFINITION
class WhatNetThai(nn.Module):
    """
    WhatNet-Thai  –  PyTorch port.

    Architecture:
      Stem : three parallel convolutions (3×3, 1×5, 5×1) → channel attention
      Enc1 : DenseResBlock 96  → 32×32
      Enc2 : DenseResBlock 192 → 16×16
      Enc3 : DenseResBlock 384 →  8× 8
      Head : multi-scale GAP fusion → AFC + dense head → gated logit blend
    """
    def __init__(self, num_classes: int, image_size: int = 64,
                 head_units: int = 256):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_t = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 32, (1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 32, (5, 1), padding=(2, 0), bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_attn = ChannelAttention(96)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(96, 96, 1, bias=False),
            nn.BatchNorm2d(96), nn.GELU(),
        )

        # Encoder
        self.enc1 = DenseResBlock(96,  96)
        self.enc2 = DenseResBlock(96,  192)
        self.enc3 = DenseResBlock(192, 384)

        F_DIM = 96 + 192 + 384   # 672

        # Adaptive Filter Capsule
        self.afc = AdaptiveFilterCapsule(F_DIM, K)

        # Dense head
        self.head_dense  = nn.Linear(F_DIM, head_units, bias=False)
        self.head_ln     = nn.LayerNorm(head_units)
        self.head_logits = nn.Linear(head_units, K)

        # Gated fusion
        self.gate = nn.Linear(K * 2, 2)

    def forward(self, x):
        # Stem
        t    = self.stem_t(x)
        h    = self.stem_h(x)
        v    = self.stem_v(x)
        stem = torch.cat([t, h, v], dim=1)
        stem = self.stem_attn(stem)
        stem = self.stem_proj(stem)

        # Encoder
        e1 = self.enc1(stem)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        # Multi-scale GAP
        gap1  = e1.mean(dim=(-2, -1))
        gap2  = e2.mean(dim=(-2, -1))
        gap3  = e3.mean(dim=(-2, -1))
        fused = torch.cat([gap1, gap2, gap3], dim=1)

        # AFC branch
        afc_out = self.afc(fused)

        # Dense head branch
        x_ = F.gelu(self.head_ln(self.head_dense(fused)))
        x_ = self.head_logits(x_)

        # Gated fusion
        combined   = torch.cat([x_, afc_out], dim=1)
        gate       = F.softmax(self.gate(combined), dim=1)
        x_scaled   = x_      * gate[:, 0:1]
        afc_scaled = afc_out * gate[:, 1:2]
        logits     = x_scaled + afc_scaled
        return logits

#  6. LR SCHEDULE  (cosine annealing)
def cosine_annealing_lambda(total_steps: int, base_lr: float):
    def lr_lambda(current_step: int):
        cosine = 0.5 * (1.0 + math.cos(math.pi * current_step / total_steps))
        return max(cosine, 1e-6 / base_lr)
    return lr_lambda

#  7. TRAINING & EVALUATION HELPERS
def compute_macro_f1(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            preds  = model(images).argmax(dim=1).cpu().numpy()
            lbls   = labels.numpy()
            for c in range(NUM_CLASSES):
                tp[c] += np.sum((preds == c) & (lbls == c))
                fp[c] += np.sum((preds == c) & (lbls != c))
                fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

def evaluate(model: nn.Module, loader: DataLoader,
             criterion: nn.Module) -> tuple:
    """Returns (avg_loss, accuracy_pct)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits  = model(images)
            loss    = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            preds   = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
    return total_loss / total, correct / total * 100.0

#  8. TRAINING LOOP
steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

_COL_BAR_W = 20

def train_one_epoch(model, loader, optimizer, scheduler, criterion, epoch, total_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total * 100.0


def train_model(model: nn.Module, name: str) -> dict:
    model = model.to(DEVICE)
    print_model_summary(model, name)

    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    optimizer = AdamW(model.parameters(),
                      lr=CFG["lr"],
                      weight_decay=CFG["weight_decay"])
    scheduler = LambdaLR(optimizer, cosine_annealing_lambda(total_steps, CFG["lr"]))

    best_val_acc = 0.0
    best_state   = None
    patience     = 15
    no_improve   = 0
    history      = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0 = time.time()

    for epoch in range(CFG["epochs"]):
        ep_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            epoch, CFG["epochs"])
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc / 100.0)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc / 100.0)

        ep_num  = epoch + 1
        pct     = ep_num / CFG["epochs"]
        filled  = int(_COL_BAR_W * pct)
        bar     = "█" * filled + "░" * (_COL_BAR_W - filled)
        lr_val  = scheduler.get_last_lr()[0]
        elapsed = time.time() - ep_start
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{CFG["epochs"]}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={train_loss:.4f}', 'white')}  "
            f"{_c(f'acc={train_acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < train_acc else 'green')}  "
            f"{_c(f'lr={lr_val:.2e}', 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            ckpt_path    = os.path.join(CFG["results_dir"], f"{name}_best.pt")
            torch.save(best_state, ckpt_path)
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  {_c('Early stopping triggered.', 'yellow')} "
                      f"(no val_acc improvement for {patience} epochs)")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - t0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val_acc:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    return model, history

#  9. RUN
MODELS = {
    "WhatNet-Thai": lambda: WhatNetThai(NUM_CLASSES, IMG),
}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

trained_models = {}
all_histories  = {}

for name, model_fn in MODELS.items():
    model, history = train_model(model_fn(), name)
    trained_models[name] = model
    all_histories[name]  = history

#  10. FINAL TEST-SET EVALUATION
criterion_eval = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
results = {}

for name, model in trained_models.items():
    test_loss, test_acc = evaluate(model, test_loader, criterion_eval)
    macro_f1            = compute_macro_f1(model, test_loader)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    sum(p.numel() for p in model.parameters()),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

#  11. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

[INFO] Using device: cuda
[INFO] Extracting /teamspace/studios/this_studio/thai-dataset.zip → /teamspace/studios/this_studio/thai-dataset-extracted …
[INFO] Extraction complete.
[INFO] train_dir = /teamspace/studios/this_studio/thai-dataset-extracted/thai-dataset/train
[INFO] test_dir  = /teamspace/studios/this_studio/thai-dataset-extracted/thai-dataset/test
[INFO] NUM_CLASSES = 68
[INFO] Classes (first 5): ['00-161-A1-KO KAI', '01-162-A2-KHO KHAI', '02-163-A3-KHO KHUAT', '03-164-A4-KHO KHWAI', '04-165-A5-KHO KHON'] …
[INFO] Short labels (first 5): ['KO KAI', 'KHO KHAI', 'KHO KHUAT', 'KHO KHWAI', 'KHO KHON'] …
[INFO] Building dataset objects …
[INFO] Train: 56,995 | Val: 6,332 | Test: 13,600

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [Thai folder dataset, 68 classes]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatN

In [ ]:
[INFO] Using device: cuda
[INFO] Extracting /teamspace/studios/this_studio/thai-dataset.zip → /teamspace/studios/this_studio/thai-dataset-extracted …
[INFO] Extraction complete.
[INFO] train_dir = /teamspace/studios/this_studio/thai-dataset-extracted/thai-dataset/train
[INFO] test_dir  = /teamspace/studios/this_studio/thai-dataset-extracted/thai-dataset/test
[INFO] NUM_CLASSES = 68
[INFO] Classes (first 5): ['00-161-A1-KO KAI', '01-162-A2-KHO KHAI', '02-163-A3-KHO KHUAT', '03-164-A4-KHO KHWAI', '04-165-A5-KHO KHON'] …
[INFO] Short labels (first 5): ['KO KAI', 'KHO KHAI', 'KHO KHUAT', 'KHO KHWAI', 'KHO KHON'] …
[INFO] Building dataset objects …
[INFO] Train: 56,995 | Val: 6,332 | Test: 13,600
[96m
══════════════════════════════════════════════════════════════════════[0m
[1m  Starting benchmark: 1 model(s) × 100 epochs  [Thai folder dataset, 68 classes][0m
[96m══════════════════════════════════════════════════════════════════════
[0m

[96m╔══════════════════════════════════════════════════════════════╗[0m
[96m[1m║  WhatNet-Thai  –  Parameter Summary                          ║[0m
[96m╠══════════════════════════════════════════════════════════════╣[0m
[92m║  Trainable params                      :         11,850,346  ║[0m
[90m║  Non-trainable params                  :                  0  ║[0m
[1m[97m║  Total params                          :         11,850,346  ║[0m
[96m╚══════════════════════════════════════════════════════════════╝[0m

[1m[96m  ▶ Training:[0m [93mWhatNet-Thai[0m
  [90mEpoch   1/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  1.0%[0m  [97mloss=2.4142[0m  [92macc=50.51%[0m  [97mval_loss=1.5622[0m  [92mval_acc=74.64%[0m  [94mlr=5.00e-04[0m  [90m[33.8s][0m
  [90mEpoch   2/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  2.0%[0m  [97mloss=1.2794[0m  [92macc=84.25%[0m  [97mval_loss=1.1987[0m  [92mval_acc=87.14%[0m  [94mlr=5.00e-04[0m  [90m[32.9s][0m
  [90mEpoch   3/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  3.0%[0m  [97mloss=1.1371[0m  [92macc=88.88%[0m  [97mval_loss=1.0070[0m  [92mval_acc=93.79%[0m  [94mlr=4.99e-04[0m  [90m[32.9s][0m
  [90mEpoch   4/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  4.0%[0m  [97mloss=1.0617[0m  [92macc=90.87%[0m  [97mval_loss=1.0002[0m  [92mval_acc=93.48%[0m  [94mlr=4.98e-04[0m  [90m[32.9s][0m
  [90mEpoch   5/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  5.0%[0m  [97mloss=1.0151[0m  [92macc=92.18%[0m  [97mval_loss=0.9732[0m  [92mval_acc=94.14%[0m  [94mlr=4.97e-04[0m  [90m[32.9s][0m
  [90mEpoch   6/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  6.0%[0m  [97mloss=0.9810[0m  [92macc=92.98%[0m  [97mval_loss=0.9665[0m  [92mval_acc=93.65%[0m  [94mlr=4.96e-04[0m  [90m[33.0s][0m
  [90mEpoch   7/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  7.0%[0m  [97mloss=0.9577[0m  [92macc=93.62%[0m  [97mval_loss=0.9296[0m  [92mval_acc=95.20%[0m  [94mlr=4.94e-04[0m  [90m[33.0s][0m
  [90mEpoch   8/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  8.0%[0m  [97mloss=0.9435[0m  [92macc=93.82%[0m  [97mval_loss=0.9063[0m  [92mval_acc=95.21%[0m  [94mlr=4.92e-04[0m  [90m[33.0s][0m
  [90mEpoch   9/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  9.0%[0m  [97mloss=0.9247[0m  [92macc=94.36%[0m  [97mval_loss=0.9077[0m  [92mval_acc=95.21%[0m  [94mlr=4.90e-04[0m  [90m[32.9s][0m
  [90mEpoch  10/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 10.0%[0m  [97mloss=0.9151[0m  [92macc=94.63%[0m  [97mval_loss=0.8957[0m  [92mval_acc=95.66%[0m  [94mlr=4.88e-04[0m  [90m[32.9s][0m
  [90mEpoch  11/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 11.0%[0m  [97mloss=0.9011[0m  [92macc=95.05%[0m  [97mval_loss=0.8820[0m  [92mval_acc=96.15%[0m  [94mlr=4.85e-04[0m  [90m[33.0s][0m
  [90mEpoch  12/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 12.0%[0m  [97mloss=0.8934[0m  [92macc=95.19%[0m  [97mval_loss=6.7119[0m  [93mval_acc=5.53%[0m  [94mlr=4.82e-04[0m  [90m[33.1s][0m
  [90mEpoch  13/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 13.0%[0m  [97mloss=0.8865[0m  [92macc=95.38%[0m  [97mval_loss=0.8803[0m  [92mval_acc=95.74%[0m  [94mlr=4.79e-04[0m  [90m[32.9s][0m
  [90mEpoch  14/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 14.0%[0m  [97mloss=0.8811[0m  [92macc=95.49%[0m  [97mval_loss=0.8796[0m  [92mval_acc=95.78%[0m  [94mlr=4.76e-04[0m  [90m[32.9s][0m
  [90mEpoch  15/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 15.0%[0m  [97mloss=0.8704[0m  [92macc=95.82%[0m  [97mval_loss=0.8728[0m  [92mval_acc=96.08%[0m  [94mlr=4.73e-04[0m  [90m[32.9s][0m
  [90mEpoch  16/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 16.0%[0m  [97mloss=0.8676[0m  [92macc=95.87%[0m  [97mval_loss=0.8721[0m  [93mval_acc=95.67%[0m  [94mlr=4.69e-04[0m  [90m[33.0s][0m
  [90mEpoch  17/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 17.0%[0m  [97mloss=0.8616[0m  [92macc=96.01%[0m  [97mval_loss=0.8707[0m  [92mval_acc=96.02%[0m  [94mlr=4.65e-04[0m  [90m[33.0s][0m
  [90mEpoch  18/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 18.0%[0m  [97mloss=0.8583[0m  [92macc=96.15%[0m  [97mval_loss=0.8693[0m  [93mval_acc=96.08%[0m  [94mlr=4.61e-04[0m  [90m[33.0s][0m
  [90mEpoch  19/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 19.0%[0m  [97mloss=0.8532[0m  [92macc=96.19%[0m  [97mval_loss=0.8723[0m  [93mval_acc=96.00%[0m  [94mlr=4.57e-04[0m  [90m[32.9s][0m
  [90mEpoch  20/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 20.0%[0m  [97mloss=0.8488[0m  [92macc=96.43%[0m  [97mval_loss=0.8635[0m  [93mval_acc=96.16%[0m  [94mlr=4.52e-04[0m  [90m[33.0s][0m
  [90mEpoch  21/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 21.0%[0m  [97mloss=0.8462[0m  [92macc=96.42%[0m  [97mval_loss=0.8791[0m  [93mval_acc=95.69%[0m  [94mlr=4.48e-04[0m  [90m[33.0s][0m
  [90mEpoch  22/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 22.0%[0m  [97mloss=0.8403[0m  [92macc=96.60%[0m  [97mval_loss=0.8679[0m  [93mval_acc=95.74%[0m  [94mlr=4.43e-04[0m  [90m[33.0s][0m
  [90mEpoch  23/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 23.0%[0m  [97mloss=0.8376[0m  [92macc=96.75%[0m  [97mval_loss=0.8685[0m  [93mval_acc=95.96%[0m  [94mlr=4.38e-04[0m  [90m[33.0s][0m
  [90mEpoch  24/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 24.0%[0m  [97mloss=0.8357[0m  [92macc=96.70%[0m  [97mval_loss=0.8695[0m  [93mval_acc=95.94%[0m  [94mlr=4.32e-04[0m  [90m[32.9s][0m
  [90mEpoch  25/100[0m  [96m█████░░░░░░░░░░░░░░░[0m [93m 25.0%[0m  [97mloss=0.8311[0m  [92macc=96.96%[0m  [97mval_loss=0.8631[0m  [93mval_acc=95.97%[0m  [94mlr=4.27e-04[0m  [90m[33.0s][0m
  [90mEpoch  26/100[0m  [96m█████░░░░░░░░░░░░░░░[0m [93m 26.0%[0m  [97mloss=0.8287[0m  [92macc=96.99%[0m  [97mval_loss=0.8562[0m  [93mval_acc=96.46%[0m  [94mlr=4.21e-04[0m  [90m[32.9s][0m
  [90mEpoch  27/100[0m  [96m█████░░░░░░░░░░░░░░░[0m [93m 27.0%[0m  [97mloss=0.8254[0m  [92macc=97.10%[0m  [97mval_loss=0.8553[0m  [93mval_acc=96.42%[0m  [94mlr=4.15e-04[0m  [90m[32.9s][0m
  [90mEpoch  28/100[0m  [96m█████░░░░░░░░░░░░░░░[0m [93m 28.0%[0m  [97mloss=0.8225[0m  [92macc=97.17%[0m  [97mval_loss=0.8608[0m  [93mval_acc=96.23%[0m  [94mlr=4.09e-04[0m  [90m[33.0s][0m
  [90mEpoch  29/100[0m  [96m█████░░░░░░░░░░░░░░░[0m [93m 29.0%[0m  [97mloss=0.8177[0m  [92macc=97.35%[0m  [97mval_loss=0.8567[0m  [93mval_acc=96.21%[0m  [94mlr=4.03e-04[0m  [90m[32.9s][0m
  [90mEpoch  30/100[0m  [96m██████░░░░░░░░░░░░░░[0m [93m 30.0%[0m  [97mloss=0.8190[0m  [92macc=97.29%[0m  [97mval_loss=0.8539[0m  [93mval_acc=96.21%[0m  [94mlr=3.97e-04[0m  [90m[32.9s][0m
  [90mEpoch  31/100[0m  [96m██████░░░░░░░░░░░░░░[0m [93m 31.0%[0m  [97mloss=0.8162[0m  [92macc=97.29%[0m  [97mval_loss=0.8529[0m  [93mval_acc=96.38%[0m  [94mlr=3.91e-04[0m  [90m[33.0s][0m
  [90mEpoch  32/100[0m  [96m██████░░░░░░░░░░░░░░[0m [93m 32.0%[0m  [97mloss=0.8132[0m  [92macc=97.46%[0m  [97mval_loss=0.8531[0m  [93mval_acc=96.38%[0m  [94mlr=3.84e-04[0m  [90m[33.0s][0m
  [90mEpoch  33/100[0m  [96m██████░░░░░░░░░░░░░░[0m [93m 33.0%[0m  [97mloss=0.8092[0m  [92macc=97.63%[0m  [97mval_loss=0.8600[0m  [93mval_acc=96.10%[0m  [94mlr=3.77e-04[0m  [90m[32.9s][0m
  [90mEpoch  34/100[0m  [96m██████░░░░░░░░░░░░░░[0m [93m 34.0%[0m  [97mloss=0.8089[0m  [92macc=97.59%[0m  [97mval_loss=0.8536[0m  [93mval_acc=96.37%[0m  [94mlr=3.70e-04[0m  [90m[32.9s][0m
  [90mEpoch  35/100[0m  [96m███████░░░░░░░░░░░░░[0m [93m 35.0%[0m  [97mloss=0.8058[0m  [92macc=97.77%[0m  [97mval_loss=0.8538[0m  [93mval_acc=96.32%[0m  [94mlr=3.63e-04[0m  [90m[32.9s][0m
  [90mEpoch  36/100[0m  [96m███████░░░░░░░░░░░░░[0m [93m 36.0%[0m  [97mloss=0.8026[0m  [92macc=97.84%[0m  [97mval_loss=0.8641[0m  [93mval_acc=96.05%[0m  [94mlr=3.56e-04[0m  [90m[32.9s][0m
  [90mEpoch  37/100[0m  [96m███████░░░░░░░░░░░░░[0m [93m 37.0%[0m  [97mloss=0.8018[0m  [92macc=97.78%[0m  [97mval_loss=0.8513[0m  [93mval_acc=96.18%[0m  [94mlr=3.49e-04[0m  [90m[33.0s][0m
  [90mEpoch  38/100[0m  [96m███████░░░░░░░░░░░░░[0m [93m 38.0%[0m  [97mloss=0.8000[0m  [92macc=97.84%[0m  [97mval_loss=0.8487[0m  [93mval_acc=96.62%[0m  [94mlr=3.42e-04[0m  [90m[33.0s][0m
  [90mEpoch  39/100[0m  [96m███████░░░░░░░░░░░░░[0m [93m 39.0%[0m  [97mloss=0.7967[0m  [92macc=97.98%[0m  [97mval_loss=0.8584[0m  [93mval_acc=96.13%[0m  [94mlr=3.35e-04[0m  [90m[33.0s][0m
  [90mEpoch  40/100[0m  [96m████████░░░░░░░░░░░░[0m [93m 40.0%[0m  [97mloss=0.7954[0m  [92macc=97.99%[0m  [97mval_loss=0.8549[0m  [93mval_acc=96.40%[0m  [94mlr=3.27e-04[0m  [90m[32.9s][0m
  [90mEpoch  41/100[0m  [96m████████░░░░░░░░░░░░[0m [93m 41.0%[0m  [97mloss=0.7947[0m  [92macc=98.00%[0m  [97mval_loss=0.8626[0m  [93mval_acc=95.94%[0m  [94mlr=3.20e-04[0m  [90m[32.9s][0m
  [90mEpoch  42/100[0m  [96m████████░░░░░░░░░░░░[0m [93m 42.0%[0m  [97mloss=0.7909[0m  [92macc=98.17%[0m  [97mval_loss=0.8588[0m  [93mval_acc=96.23%[0m  [94mlr=3.12e-04[0m  [90m[32.9s][0m
  [90mEpoch  43/100[0m  [96m████████░░░░░░░░░░░░[0m [93m 43.0%[0m  [97mloss=0.7910[0m  [92macc=98.20%[0m  [97mval_loss=0.8559[0m  [93mval_acc=96.07%[0m  [94mlr=3.05e-04[0m  [90m[32.9s][0m
  [90mEpoch  44/100[0m  [96m████████░░░░░░░░░░░░[0m [93m 44.0%[0m  [97mloss=0.7878[0m  [92macc=98.27%[0m  [97mval_loss=0.8566[0m  [93mval_acc=96.27%[0m  [94mlr=2.97e-04[0m  [90m[32.9s][0m
  [90mEpoch  45/100[0m  [96m█████████░░░░░░░░░░░[0m [93m 45.0%[0m  [97mloss=0.7862[0m  [92macc=98.39%[0m  [97mval_loss=0.8582[0m  [93mval_acc=96.23%[0m  [94mlr=2.89e-04[0m  [90m[33.0s][0m
  [90mEpoch  46/100[0m  [96m█████████░░░░░░░░░░░[0m [93m 46.0%[0m  [97mloss=0.7843[0m  [92macc=98.43%[0m  [97mval_loss=0.8579[0m  [93mval_acc=96.13%[0m  [94mlr=2.81e-04[0m  [90m[33.0s][0m
  [90mEpoch  47/100[0m  [96m█████████░░░░░░░░░░░[0m [93m 47.0%[0m  [97mloss=0.7816[0m  [92macc=98.55%[0m  [97mval_loss=0.8561[0m  [93mval_acc=96.46%[0m  [94mlr=2.74e-04[0m  [90m[32.9s][0m
  [90mEpoch  48/100[0m  [96m█████████░░░░░░░░░░░[0m [93m 48.0%[0m  [97mloss=0.7820[0m  [92macc=98.45%[0m  [97mval_loss=0.8575[0m  [93mval_acc=96.18%[0m  [94mlr=2.66e-04[0m  [90m[33.0s][0m
  [90mEpoch  49/100[0m  [96m█████████░░░░░░░░░░░[0m [93m 49.0%[0m  [97mloss=0.7795[0m  [92macc=98.58%[0m  [97mval_loss=0.8569[0m  [93mval_acc=96.32%[0m  [94mlr=2.58e-04[0m  [90m[33.0s][0m
  [90mEpoch  50/100[0m  [96m██████████░░░░░░░░░░[0m [93m 50.0%[0m  [97mloss=0.7769[0m  [92macc=98.68%[0m  [97mval_loss=0.8639[0m  [93mval_acc=95.96%[0m  [94mlr=2.50e-04[0m  [90m[33.0s][0m
  [90mEpoch  51/100[0m  [96m██████████░░░░░░░░░░[0m [93m 51.0%[0m  [97mloss=0.7746[0m  [92macc=98.74%[0m  [97mval_loss=0.8574[0m  [93mval_acc=96.42%[0m  [94mlr=2.42e-04[0m  [90m[33.0s][0m
  [90mEpoch  52/100[0m  [96m██████████░░░░░░░░░░[0m [93m 52.0%[0m  [97mloss=0.7749[0m  [92macc=98.75%[0m  [97mval_loss=0.8624[0m  [93mval_acc=96.16%[0m  [94mlr=2.34e-04[0m  [90m[33.0s][0m
  [90mEpoch  53/100[0m  [96m██████████░░░░░░░░░░[0m [93m 53.0%[0m  [97mloss=0.7730[0m  [92macc=98.87%[0m  [97mval_loss=0.8601[0m  [93mval_acc=96.24%[0m  [94mlr=2.26e-04[0m  [90m[33.0s][0m
  [93mEarly stopping triggered.[0m (no val_acc improvement for 15 epochs)

  [92m[1m✔ Done:[0m best val acc = [92m96.62%[0m  wall time = [90m1749s[0m

[96m[1m╔══════════════════════════════════════════════════════════════════════╗[0m
[96m[1m║  FINAL TEST-SET COMPARISON                                           ║[0m
[96m╠════════════════════════╦════════════╦════════════╦════════════╦══════╣[0m
[96m[1m║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║[0m
[96m╠════════════════════════╬════════════╬════════════╬════════════╬══════╣[0m
[92m[1m║★ WhatNet-Thai          ║11,850,346 ║     96.93%║     96.92%║0.831 ║[0m
[96m╚════════════════════════╩════════════╩════════════╩════════════╩══════╝[0m
[92m[1m
  ★  Winner: WhatNet-Thai  (96.93% test accuracy)
[0m
[INFO] Results   → ./results/thai_results.json
[INFO] Histories → ./results/thai_histories.json
[92m[1m
[DONE] Thai folder-dataset benchmark complete.
[0m

In [7]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple


def get_scaffold_activations(
    model: nn.Module,
    image: torch.Tensor,
    stage_layer_names: List[str]
) -> List[np.ndarray]:
    """
    model: your trained LipiNet model (PyTorch)
    image: single input image, shape (1, 1, H, W) — NCHW format
    stage_layer_names: list of layer names for scaffold
                       injection points at enc1, enc2, enc3
    """
    activations = []
    hooks = []

    def make_hook(storage: list):
        def hook_fn(module, input, output):
            storage.append(output.detach().cpu().numpy())
        return hook_fn

    # Register forward hooks on the named layers
    for name in stage_layer_names:
        # Support both direct attributes and nested (e.g. "encoder.layer1")
        layer = model
        for part in name.split("."):
            layer = getattr(layer, part)

        hook = layer.register_forward_hook(make_hook(activations))
        hooks.append(hook)

    model.eval()
    with torch.no_grad():
        model(image)

    # Remove hooks after inference
    for hook in hooks:
        hook.remove()

    return activations


def visualize_psi_cross_script(
    models_and_images: List[Tuple[nn.Module, torch.Tensor]],
    stage_layer_names_per_model: List[List[str]],
    script_names: List[str]
) -> None:
    """
    models_and_images: list of (model, sample_image) tuples
                       images should be shape (1, 1, H, W)
    script_names: ['Devanagari', 'Arabic', 'Kannada', 'Thai']
    """
    n_scripts = len(script_names)
    n_stages = 4  # stem + enc1 + enc2 + enc3

    fig, axes = plt.subplots(
        n_scripts, n_stages,
        figsize=(16, 4 * n_scripts)
    )

    # Ensure axes is always 2D even for a single script
    if n_scripts == 1:
        axes = axes[np.newaxis, :]

    col_titles = [
        'Scaffold (Stem)',
        'PSI — Enc Stage 1',
        'PSI — Enc Stage 2',
        'PSI — Enc Stage 3'
    ]

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(
            title, fontsize=13, fontweight='bold', pad=10
        )

    for row, (script, (model, image), layer_names) in enumerate(
        zip(script_names, models_and_images, stage_layer_names_per_model)
    ):
        activations = get_scaffold_activations(model, image, layer_names)

        axes[row, 0].set_ylabel(
            script, fontsize=12, fontweight='bold',
            rotation=90, labelpad=10
        )

        for col, act in enumerate(activations):
            # act shape: (1, C, H, W) — remove batch dim → (C, H, W)
            act_map = act[0]

            if act_map.ndim == 3:
                # Average over channel dim (C, H, W) → (H, W)
                act_map = np.mean(act_map, axis=0)

            # L2 normalise per stage for comparability
            act_map = act_map / (np.linalg.norm(act_map) + 1e-8)

            im = axes[row, col].imshow(
                act_map,
                cmap='viridis',
                interpolation='nearest'
            )
            axes[row, col].axis('off')
            plt.colorbar(im, ax=axes[row, col],
                         fraction=0.046, pad=0.04)

    plt.suptitle(
        'Cross-Script PSI Scaffold Activations\n'
        'Rows: Scripts | Columns: Encoder Stages',
        fontsize=14, fontweight='bold', y=1.02
    )

    plt.tight_layout()
    plt.savefig(
        'psi_cross_script_visualization.pdf',
        bbox_inches='tight', dpi=300
    )
    plt.savefig(
        'psi_cross_script_visualization.png',
        bbox_inches='tight', dpi=300
    )
    plt.show()
    print("Saved to psi_cross_script_visualization.pdf/.png")

In [ ]:
import tensorflow as tf

# 1. Save weights only
model.save_weights('my_weights.h5')

# 2. Save entire model (architecture + weights)
model.save('full_model.h5')

# To load them back later:
# Weights only (requires existing model architecture)
model.load_weights('my_weights.h5')

# Entire model
new_model = tf.keras.models.load_model('full_model.h5')


In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_weights.h5',
    save_weights_only=True,
    monitor='val_accuracy',
    save_best_only=True
)

model.fit(train_data, epochs=10, callbacks=[checkpoint])


In [ ]:
import torch
# Assuming 'model' is your neural network
torch.save
(model.state_dict(), 'model_weights.pth')

model = MyModelClass() # Must be the same architecture
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model.eval() # Essential for dropout/batchnorm layers


In [ ]:
torch.save
({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}, 'checkpoint.pth')
